# Week 91, Day 1: Performance

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w091d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('profile.txt', 'w', encoding='utf-8') as f:
    f.write('         116401 function calls (116201 primitive calls) in 0.690 seconds\n\n   Ordered by: cumulative time\n\n   ncalls  tottime  percall  cumtime  percall filename:lineno(function)\n        1    0.002    0.002    0.690    0.690 cli.py:12(main)\n      120    0.012    0.000    0.680    0.006 planner.py:40(plan)\n      120    0.046    0.000    0.660    0.006 waymark_domain.py:48(shortest)\n    35520    0.420    0.000    0.560    0.000 waymark_domain.py:44(climb)\n9600/9400    0.140    0.000    0.140    0.000 {built-in method builtins.sorted}\n    71040    0.070    0.000    0.070    0.000 {built-in method builtins.max}\n')
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')

**Predict before you run:** what will this print?



In [ ]:
runs_ms = [41.0, 39.5, 40.2, 212.0, 40.8]
print(sum(runs_ms) / len(runs_ms), sorted(runs_ms)[len(runs_ms) // 2])

> **Key idea.** Time with `perf_counter`, repeated, warmed up, on realistic data; report the median and the nearest-rank p95. Profile to find where: `tottime` finds the hot spot, `cumtime` the slow path, and `ncalls` often explains both. Fix in order: do less work (precompute, once), use the right structure (dictionaries, not pairwise comparisons), stream large inputs, and only then small tricks. Invalidate anything precomputed. Write budgets as numbers and check them.

### Exercise: Summarising timings

Write `timing_summary(samples)`: given a list of run times in milliseconds, return a dict with `runs` (the count), `min`, `median`, `p95` (nearest rank: the value at position `ceil(0.95 * n)` counting from 1) and `max`, each time rounded to 1 decimal place. Raise `ValueError("no samples")` for an empty list.

In [ ]:
import math

def timing_summary(samples):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_summary():\n    \"median and p95\"\n    assert timing_summary([41.0, 39.5, 40.2, 212.0, 40.8]) == {\"runs\": 5, \"min\": 39.5, \"median\": 40.8, \"p95\": 212.0, \"max\": 212.0}\n    assert timing_summary([float(i) for i in range(1, 21)]) == {\"runs\": 20, \"min\": 1.0, \"median\": 10.5, \"p95\": 19.0, \"max\": 20.0}\n    assert timing_summary([3.14159]) == {\"runs\": 1, \"min\": 3.1, \"median\": 3.1, \"p95\": 3.1, \"max\": 3.1}\n    try:\n        timing_summary([])\n    except ValueError as e:\n        assert str(e) == \"no samples\"\n    else:\n        raise AssertionError(\"empty\")")

### Exercise: Reading a profile

Write `profile_rows(text)`: parse a cProfile table like `profile.txt`, returning a list of dicts with `name` (the last column, exactly as written), `ncalls` (an int: the first number when it's written `a/b`), `tottime` and `cumtime` (floats), for every row after the header line that starts with `ncalls`. Then `top(text, key="tottime", n=3)`: the `n` rows with the largest `key` (`"tottime"`, `"cumtime"` or `"ncalls"`; anything else raises `ValueError`), as `(name, value)` pairs, largest first and ties by name.

In [ ]:
def profile_rows(text):
    ___

def top(text, key="tottime", n=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rows():\n    \"every row\"\n    rows = profile_rows(open(\"profile.txt\").read())\n    assert len(rows) == 6\n    assert rows[0] == {\"name\": \"cli.py:12(main)\", \"ncalls\": 1, \"tottime\": 0.002, \"cumtime\": 0.69}\n    assert rows[4] == {\"name\": \"{built-in method builtins.sorted}\", \"ncalls\": 9600, \"tottime\": 0.14, \"cumtime\": 0.14}\n\ndef test_top():\n    \"hot spots and slow paths\"\n    text = open(\"profile.txt\").read()\n    assert top(text) == [(\"waymark_domain.py:44(climb)\", 0.42), (\"{built-in method builtins.sorted}\", 0.14),\n                         (\"{built-in method builtins.max}\", 0.07)]\n    assert top(text, \"cumtime\", 2) == [(\"cli.py:12(main)\", 0.69), (\"planner.py:40(plan)\", 0.68)]\n    assert top(text, \"ncalls\", 1) == [(\"{built-in method builtins.max}\", 71040)]\n    try:\n        top(text, \"percall\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"percall\")")

### Exercise: A distance table, computed once

Write `DistanceTable(network)`: on creation, run Dijkstra **once from each junction** over `network.paths` (a dict from junction to `{neighbour: km}`), counting the runs in `self.runs`. `km(a, b)` returns the shortest distance rounded to 2 decimal places, or `None` if there's no route; it never searches again. `refresh()` rebuilds the table (after the map changes).

In [ ]:
import heapq

class DistanceTable:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_map import sample_network\n\ndef test_table():\n    \"one run per junction\"\n    network = sample_network()\n    table = DistanceTable(network)\n    assert table.runs == 8\n    for a in (\"Mill\", \"Ridge\", \"Lake\"):\n        for b in (\"Weir\", \"Ridge\", \"Wood\"):\n            assert table.km(a, b) == network.shortest(a, b).km\n    for _ in range(100):\n        table.km(\"Mill\", \"Ridge\")\n    assert table.runs == 8 and table.km(\"Mill\", \"Island\") is None and table.km(\"Mill\", \"Mill\") == 0\n\ndef test_refresh():\n    \"a table that knows about new paths\"\n    network = sample_network()\n    table = DistanceTable(network)\n    network.add_path(\"Mill\", \"Ridge\", 4.5)\n    assert table.km(\"Mill\", \"Ridge\") == 6.0\n    table.refresh()\n    assert table.km(\"Mill\", \"Ridge\") == 4.5 and table.runs == 16")

### Exercise: Repeated walks, in linear time

A walker who logs the same walk twice skews every statistic. Write `find_repeats(walks)`: `walks` is a list of dicts with `day`, `km` and `minutes`; a walk is a **repeat** if an earlier walk has the same three values. Return a list of `(first, repeat)` index pairs (the index of the first such walk, and of the repeat), in order of the repeat. It must handle 30,000 walks in well under a second, so compare every pair is out.

In [ ]:
def find_repeats(walks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\n\ndef test_repeats():\n    \"the same walk twice\"\n    walks = [{\"day\": \"2026-01-04\", \"km\": 6.0, \"minutes\": 105}, {\"day\": \"2026-01-18\", \"km\": 7.5, \"minutes\": 98},\n             {\"day\": \"2026-01-04\", \"km\": 6.0, \"minutes\": 105}, {\"day\": \"2026-01-04\", \"km\": 6.0, \"minutes\": 104},\n             {\"day\": \"2026-01-04\", \"km\": 6.0, \"minutes\": 105}, {\"day\": \"2026-01-18\", \"km\": 7.5, \"minutes\": 98}]\n    assert find_repeats(walks) == [(0, 2), (0, 4), (1, 5)]\n    assert find_repeats([]) == []\n\ndef test_fast():\n    \"30,000 walks\"\n    walks = [{\"day\": f\"2026-{1 + i % 12:02d}-{1 + i % 28:02d}\", \"km\": float(i % 997), \"minutes\": i % 991} for i in range(30_000)]\n    start = time.perf_counter()\n    found = find_repeats(walks)\n    assert time.perf_counter() - start < 1.0, \"too slow: compare by key, not every pair\"\n    assert len(found) == 30_000 - len({(w[\"day\"], w[\"km\"], w[\"minutes\"]) for w in walks})")

### Exercise: Streaming the walk log

Write a **generator** `iter_walks(lines)` that takes any iterable of CSV lines (`day,km,minutes`, with a header line `day,km,minutes` first and perhaps blank lines) and yields one dict per walk with `day` (a string), `km` (a float) and `minutes` (an int), reading only as many lines as it needs.

In [ ]:
def iter_walks(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import inspect\n\ndef test_walks():\n    \"one walk at a time\"\n    lines = [\"day,km,minutes\\n\", \"2026-01-04,6.0,105\\n\", \"\\n\", \"2026-01-18,7.5,98\\n\"]\n    assert list(iter_walks(lines)) == [{\"day\": \"2026-01-04\", \"km\": 6.0, \"minutes\": 105}, {\"day\": \"2026-01-18\", \"km\": 7.5, \"minutes\": 98}]\n    assert inspect.isgenerator(iter_walks(lines))\n\ndef test_lazy():\n    \"never reads more than it needs\"\n    read = []\n    def endless():\n        yield \"day,km,minutes\\n\"\n        i = 0\n        while True:\n            i += 1\n            read.append(i)\n            yield f\"2026-01-{i % 28 + 1:02d},{i}.0,{i * 12}\\n\"\n    walks = iter_walks(endless())\n    assert next(walks) == {\"day\": \"2026-01-02\", \"km\": 1.0, \"minutes\": 12}\n    assert next(walks)[\"km\"] == 2.0 and read == [1, 2]")

### Bug hunt: A cache that remembered the old map

`route_km` caches distances, but the cache isn't cleared when a path is added, so after a new direct path from Mill to Ridge it still answers 6.0. Make `add_path` here invalidate the cache, so the program prints `6.0` and then `4.5`.

In [ ]:
from waymark_map import sample_network

network = sample_network()
cache = {}

def route_km(a, b):
    if (a, b) not in cache:
        cache[(a, b)] = network.shortest(a, b).km
    return cache[(a, b)]

def add_path(a, b, km):
    network.add_path(a, b, km)

print(route_km("Mill", "Ridge"))
add_path("Mill", "Ridge", 4.5)
print(route_km("Mill", "Ridge"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"invalidate on change\"\n    expect_output(\"6.0\\n4.5\")")

### Bug hunt: A percentile one place too high

By nearest rank, the 95th percentile of the runs 1 to 20 ms is 19 ms (19 of the 20 runs were at or below it), but this prints `20.0`, the slowest run. Fix the position so it prints `19.0`.

In [ ]:
import math

def p95(samples):
    ordered = sorted(samples)
    return ordered[int(0.95 * len(ordered))]

print(p95([float(i) for i in range(1, 21)]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"position ceil(0.95 n), counting from 1\"\n    expect_output(\"19.0\")")

### Puzzle: Timing honestly

Arrange a function that warms up once, then times `repeat` runs of `fn` with the right clock, and returns the median in milliseconds. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the median of [3, 1, 2, 10]
q = ___   # the cProfile column that excludes time in called functions
r = ___   # True or False: a generator holds all the items it will produce in memory
s = ___   # Dijkstra runs a DistanceTable needs for the 8-junction sample map
t = ___   # the nearest-rank p95 of the runs 1 to 10 ms (an int)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2.5 and q == \"tottime\" and r is False, \"p, q and r\"\n    assert s == 8 and t == 10, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the performance report

Write `perf_report(profile_text, samples, budgets)`. `samples` maps a case name to a list of run times in ms; `budgets` maps case names to a p95 budget in ms, in the order to report. Return `(lines, verdict)`:

- for each budget, `"CASE: p50 X ms, p95 Y ms, budget B ms, ok"` (or `OVER` when p95 is above the budget), with X and Y to 1 decimal place (p50 is the median, p95 nearest rank), or `"CASE: not measured"` if there are no samples;
- then `"hot spots:"` and, for the three rows with the largest `tottime` (ties by name), `"  NAME: T s (P% of own time)"` with T to 3 decimal places and P the rounded percentage of the total `tottime` of all rows.

`verdict` is `"within budget"` if every case is ok, otherwise `"over budget: "` and the cases that are over or not measured, in budget order, joined by `", "`.

In [ ]:
import math

def perf_report(profile_text, samples, budgets):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SAMPLES = {\"plan Mill Ridge\": [4.1, 3.9, 4.0, 4.4, 4.2], \"plan grid 30\": [38.0, 41.5, 40.2, 39.9, 61.0, 40.0],\n           \"round walk\": [120.0, 118.5, 131.0]}\nBUDGETS = {\"plan Mill Ridge\": 10, \"plan grid 30\": 50, \"round walk\": 150}\n\ndef test_report():\n    \"budgets and hot spots\"\n    lines, verdict = perf_report(open(\"profile.txt\").read(), SAMPLES, BUDGETS)\n    assert lines == [\n        \"plan Mill Ridge: p50 4.1 ms, p95 4.4 ms, budget 10 ms, ok\",\n        \"plan grid 30: p50 40.1 ms, p95 61.0 ms, budget 50 ms, OVER\",\n        \"round walk: p50 120.0 ms, p95 131.0 ms, budget 150 ms, ok\",\n        \"hot spots:\",\n        \"  waymark_domain.py:44(climb): 0.420 s (61% of own time)\",\n        \"  {built-in method builtins.sorted}: 0.140 s (20% of own time)\",\n        \"  {built-in method builtins.max}: 0.070 s (10% of own time)\",\n    ]\n    assert verdict == \"over budget: plan grid 30\"\n\ndef test_verdicts():\n    \"within, and not measured\"\n    text = open(\"profile.txt\").read()\n    assert perf_report(text, {\"a\": [1.0]}, {\"a\": 5})[1] == \"within budget\"\n    lines, verdict = perf_report(text, {\"a\": [9.0]}, {\"a\": 5, \"b\": 5})\n    assert lines[:2] == [\"a: p50 9.0 ms, p95 9.0 ms, budget 5 ms, OVER\", \"b: not measured\"] and verdict == \"over budget: a, b\"")

## Warm-up

One run took 212 ms (perhaps the computer was busy), and it drags the mean up to 74.7 ms although a typical run takes about 40. The **median** isn't moved by one strange run. Every timing in this lesson is a set of runs, summarised.

## Measure honestly

Time with `time.perf_counter()`, the clock meant for measuring short intervals. Then follow a few rules:

- **Repeat** the measurement (five to twenty runs), and report the **median** for a typical run and the **95th percentile** (p95) for a slow one. Users notice the slow ones.
- **Warm up**: the first run often pays for imports, caches and file reads that later runs don't.
- **Time only the operation**: build the map before starting the clock, unless building it is what you're measuring.
- **Measure on realistic data**: the eight-junction sample map is too small to show anything; use Week 89's `grid_network` or a real map.

The **nearest-rank** percentile is simple and honest: sort the runs and take the one at position `ceil(0.95 * n)` (counting from 1). With 20 runs, p95 is the 19th: 19 of the 20 were at or below it.

## Reading a profile

A timer says *how long*; a **profiler** says *where*. On your own computer, `python -m cProfile -s cumtime waymark.py plan Mill Ridge` runs the program and prints a table; `profile.txt` above is one, from a draft that computed climbs inside the search while planning 120 routes. Its columns:

- `ncalls`: how many times the function was called (`9600/9400` means 9600 calls, 9400 of them not recursive).
- `tottime`: time spent **in the function itself**, not counting functions it called. This finds the hot spot.
- `cumtime`: time including everything it called. This finds the slow **path** from `main` down.

`main` has the largest `cumtime` but almost no `tottime`: it's slow only because of what it calls. `climb` has the largest `tottime`, and 35,520 calls for 120 routes is the clue: something calls it far more often than needed. A free viewer such as SnakeViz draws the same table as a picture.

## Fixing what the profile shows

Fix in this order, because the first ones win the most:

1. **Do less work**: compute something once rather than per call. Shortest routes from one junction to *every* other come from a single Dijkstra run, so a table of distances needs one run per junction, not one per question.
2. **Use the right structure**: finding repeated entries in a list by comparing every pair is quadratic; a dictionary keyed by what makes entries equal is linear.
3. **Stream**: a generator that reads a walk log line by line uses the same small memory for 100 walks or 10 million.
4. **Only then**, small tricks such as local variables or `lru_cache`, measured each time.

Anything precomputed or cached must be **invalidated** when its inputs change, or it answers yesterday's question quickly.

## Budgets

A **performance budget** is a promise written as a number: "planning a route on a 30 by 30 map takes under 50 ms at p95 on the reference machine". Budgets turn "it feels slow" into a check that passes or fails, so a slowdown is caught in review rather than by users. Keep the work counts from Week 89 in CI (they don't vary between runs) and check timings against budgets on one known machine.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Write your budgets**: three to five operations users wait for, each with a p95 in milliseconds on one named machine.
2. **Measure** each with repeated, warmed-up runs on realistic data, and **profile** the slowest with cProfile (and SnakeViz, if you like pictures).
3. **Fix one thing** the profile points at, measure again, and record before and after in the pull request.
4. **Add the check** to CI: work counts always, timings on the reference machine.

## Before you go

Tomorrow is a full security review: not one sprint's code this time, but the whole of your capstone, starting with a map of where untrusted data comes in.